# 04 · QAOA 与 VQE：变分算法

对应讲义实验 C · 约 45 分钟，另有课后作业 · 前置：01 编程基础

变分算法把问题写成哈密顿量，用参数化线路制备试探态，再交给经典优化器调整参数。本节用 QAOA 求解 Max-Cut；课后用 VQE 计算氢分子的势能曲线。

**学完本节，你将能够**
- 把组合优化问题翻译成哈密顿量；
- 用 MindSpore 训练参数化线路；
- 用暴力枚举这样的经典方法检验结果。

In [ ]:
from mqlab import check              # 先导入：同时屏蔽 MindSpore 的冗长日志
import itertools
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import mindspore as ms
from mindquantum import QubitOperator, Hamiltonian, MaxCutAnsatz, MQAnsatzOnlyLayer, Simulator

## 1. 问题：Max-Cut

把图的顶点分成两组，让跨组的边尽量多。下面这个图有 5 个顶点、6 条边。

In [ ]:
graph = nx.Graph([(0, 1), (1, 2), (2, 3), (3, 4), (0, 4), (0, 2)])
plt.figure(figsize=(3, 3))
nx.draw(graph, with_labels=True, node_color='white', edgecolors='#222', font_weight='bold')
plt.show()

### 练习 1 · 经典基线：暴力枚举

5 个顶点只有 $2^5 = 32$ 种分法，可以全部枚举。补全 `brute_force_maxcut(graph)`，返回最大割的边数。有了经典基线，才能判断量子算法的结果好不好。

In [ ]:
def brute_force_maxcut(graph):
    # 练习 1：枚举每个顶点属于哪一组（提示：itertools.product([0, 1], repeat=顶点数)）
    return ...

check.brute_force_maxcut(brute_force_maxcut)

## 2. 翻译成哈密顿量

用量子比特表示顶点的分组，$|0\rangle$、$|1\rangle$ 代表两组。对每条边 (i, j)，$Z_iZ_j$ 在两端同组时为 +1、异组时为 −1，所以

$$\text{割边数} = \frac{|E| - \langle H\rangle}{2},\qquad H = \sum_{(i,j)\in E} Z_i Z_j .$$

最小化 ⟨H⟩，就是最大化割边数。

In [ ]:
ops = QubitOperator()
for i, j in graph.edges:
    ops += QubitOperator(f'Z{i} Z{j}')     # 每条边一项
ham = Hamiltonian(ops)
print(ham)

## 3. QAOA 线路：代价层与混合层交替 p 次

代价层把问题写进相位（每条边一个 ZZ(γ) 门），混合层在候选解之间“搅拌”（每个比特一个 RX(β) 门）。`MaxCutAnsatz` 直接生成整条线路，depth = p 时共有 2p 个参数。下面是 p = 1 的样子：

In [ ]:
MaxCutAnsatz(list(graph.edges), depth=1).circuit.svg()

In [ ]:
circ = MaxCutAnsatz(list(graph.edges), depth=4).circuit
print(circ.n_qubits, '个比特，', len(circ.params_name), '个参数')

## 4. 训练：线路即网络

`get_expectation_with_grad` 给出期望值与梯度；`MQAnsatzOnlyLayer` 把它包装成 MindSpore 网络，就可以直接用 Adam 优化器训练。

In [ ]:
ms.set_seed(42)
sim = Simulator('mqvector', circ.n_qubits)
grad_ops = sim.get_expectation_with_grad(ham, circ)
net = MQAnsatzOnlyLayer(grad_ops)                 # 线路 → 可训练网络
opti = ms.nn.Adam(net.trainable_params(), learning_rate=0.05)
train_net = ms.nn.TrainOneStepCell(net, opti)

history = []
for step in range(200):
    cut = (len(graph.edges) - train_net()) / 2    # ⟨H⟩ → 割边数
    history.append(float(cut.asnumpy()[0]))
    if step % 50 == 0:
        print(f'step {step:3d}  cut = {history[-1]:.3f}')

plt.figure(figsize=(6, 3))
plt.plot(history, color='#222')
plt.axhline(5, color='#c0392b', ls='--', lw=1)    # 暴力枚举得到的最大割
plt.xlabel('step')
plt.ylabel('<cut>')
plt.show()

⟨割边数⟩ 收敛到约 4.94 < 5：线路输出的是概率分布，大部分概率集中在最优解上。

### 优化地形：一层 QAOA 只有两个参数

depth = 1 时只有 γ、β 两个参数，可以把平均割边数对所有参数都算出来，画成“地形图”。训练就是在这张图上爬坡：黑线是 Adam 的轨迹。

In [ ]:
c1 = MaxCutAnsatz(list(graph.edges), depth=1).circuit     # 参数：gamma_0、beta_0
f1 = Simulator('mqvector', 5).get_expectation_with_grad(ham, c1)
G = np.linspace(0, np.pi, 61)
B = np.linspace(-np.pi / 2, np.pi / 2, 61)
land = np.array([[(len(graph.edges) - f1(np.array([g, b]))[0].real[0, 0]) / 2 for g in G] for b in B])

ms.set_seed(42)
net1 = MQAnsatzOnlyLayer(f1)
train1 = ms.nn.TrainOneStepCell(net1, ms.nn.Adam(net1.trainable_params(), learning_rate=0.05))
path = [net1.weight.asnumpy().copy()]
for _ in range(200):
    train1()
    path.append(net1.weight.asnumpy().copy())
path = np.array(path)

plt.figure(figsize=(5, 3.4))
plt.imshow(land, origin='lower', extent=[0, np.pi, -np.pi / 2, np.pi / 2], aspect='auto',
           cmap='RdBu_r', vmin=1.5, vmax=4.5)
plt.colorbar(label='<cut>')
plt.plot(path[:, 0], path[:, 1], color='k', lw=1.5)
plt.xlabel('gamma')
plt.ylabel('beta')
plt.show()
print(f'地形最高点 {land.max():.2f}；Adam 停在 {path[-1].round(2)}')

γ = 0 或 β = 0 时，平均割边数恒为 3.0（等于随机猜）；从别处出发，梯度法可能停在次优的山峰上。层数为 p 时有 2p 个参数，地形画不出来了，但这些特征依然存在。

## 5. 采样：线路给出了哪些分法

In [ ]:
pr = dict(zip(circ.params_name, net.weight.asnumpy()))
circ.measure_all()
sim.reset()
res = sim.sampling(circ, pr=pr, shots=1000, seed=7)
top = sorted(res.data.items(), key=lambda kv: -kv[1])[:4]
print(top)

出现最多的 4 个比特串，正好是全部 4 个最优解（比特串按 q4…q0 书写，0、1 表示所在的组；`10110` 与 `01001` 是同一种分法，只是交换了两组的名字）。

### 练习 2 · 层数的影响（开放练习）

把 depth 改为 1、2、3、4，各训练 200 步，比较最终的平均割边数。层数越多，表达能力越强，但参数也越多、越难训练。

In [ ]:
# 练习 2：把上面的训练过程包装成函数，比较不同的 depth
...

### 现场改一改：先预测，再运行

每一处先写下预测，再运行：

1. 层数 1 → 4（就是练习 2）：训练后的平均割边数？　A 都是 5　B 逐渐接近 5　C 都约 4.1
2. 参数全部从 0 开始（`weight='zeros'`）：训练 200 步后，平均割边数？　A 照样到 4.94　B 停在 3.0 不动　C 报错
3. 换成 10 个顶点、15 条边的 3-正则图，depth = 4：测到最优解的概率？　A 约 96%，不变　B 约 55%　C 约 5%

In [ ]:
def train(g, depth=4, weight='normal', steps=200):
    h = Hamiltonian(sum((QubitOperator(f'Z{i} Z{j}') for i, j in g.edges), QubitOperator()))
    c = MaxCutAnsatz(list(g.edges), depth=depth).circuit
    ms.set_seed(42)
    net = MQAnsatzOnlyLayer(Simulator('mqvector', c.n_qubits).get_expectation_with_grad(h, c), weight=weight)
    step = ms.nn.TrainOneStepCell(net, ms.nn.Adam(net.trainable_params(), learning_rate=0.05))
    for _ in range(steps):
        cut = (len(g.edges) - step()) / 2
    return c, net, float(cut.asnumpy()[0])

_, net0, cut0 = train(graph, weight='zeros')
print(f'第 2 处：<cut> = {cut0:.3f}，参数 {net0.weight.asnumpy()}')

g10 = nx.random_regular_graph(3, 10, seed=1)
c10, net10, _ = train(g10)
s10 = Simulator('mqvector', 10)
s10.apply_circuit(c10, pr=dict(zip(c10.params_name, net10.weight.asnumpy())))
probs = np.abs(s10.get_qs()) ** 2
cuts = np.array([sum((k >> i & 1) != (k >> j & 1) for i, j in g10.edges) for k in range(2 ** 10)])
print(f'第 3 处：最大割 {cuts.max()}，测到最优解的概率 {probs[cuts == cuts.max()].sum():.3f}')

<details><summary>答案</summary>

1. B：4.11、4.62、4.83、4.94。层数越多越接近 5，参数也越多（2p 个）。
2. B：停在 3.0，参数一直是 0。原点处代价层和混合层都是恒等操作，状态是均匀叠加，两类参数的梯度都恰好为 0。默认的随机初始化正是为了离开这个点。
3. B：约 55%（5 个顶点时约 96%），平均割边数却已达最大值的 96%。问题越大，需要的层数越多：这个图要到 depth = 8 才超过 90%。
</details>

## 课后作业 · 用 VQE 计算氢分子的势能曲线

VQE 与 QAOA 是同一套框架：换一个哈密顿量、换一条线路，就从组合优化走到了量子化学。

1. `get_H2_ham(d)` 生成键长为 d（单位 Å）时氢分子的量子比特哈密顿量，同时给出 FCI（精确解）与 HF（平均场近似）的能量；
2. 按下图搭建 4 比特线路，参数名依次为 `a0` … `a3`、`b0` … `b3`；
3. 对 0.5–1.1 Å 的 13 个键长分别训练，画出势能曲线，与 FCI、HF 比较。

![VQE 线路](images/vqe_ansatz.svg)

> 需要 `openfermion` 与 `openfermionpyscf`，仅支持 Linux / macOS。Windows 用户请使用 MyBinder 在线环境。

In [ ]:
import tempfile
from mindquantum import Circuit, X, RY, get_qubit_hamiltonian

def get_H2_ham(d):
    """键长 d（Å）时的氢分子哈密顿量，以及 FCI、HF 能量（Hartree）。"""
    from openfermion.chem import MolecularData
    from openfermionpyscf import run_pyscf
    mol = MolecularData([('H', (0, 0, 0)), ('H', (0, 0, d))], 'sto3g', multiplicity=1,
                        data_directory=tempfile.gettempdir())
    mol = run_pyscf(mol, run_fci=True)
    return Hamiltonian(get_qubit_hamiltonian(mol)), mol.fci_energy, mol.hf_energy

In [ ]:
circ = Circuit()
# 作业 1：按图搭建线路
...
check.vqe_ansatz(circ)

In [ ]:
distances = [round(0.5 + 0.05 * i, 2) for i in range(13)]   # 0.5–1.1 Å
energies, fci, hf = [], [], []
sim = Simulator('mqvector', 4)
for d in distances:
    ham_h2, e_fci, e_hf = get_H2_ham(d)
    fci.append(e_fci)
    hf.append(e_hf)
    # 作业 2：训练线路 200 步，把收敛后的能量加入 energies
    ...

check.vqe_curve(energies, fci)

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(distances, fci, 'k-', label='FCI')
plt.plot(distances, hf, '--', color='0.6', label='HF')
if energies:
    plt.plot(distances, energies, 'o', color='#c0392b', label='VQE')
plt.xlabel('bond length (Å)')
plt.ylabel('energy (Hartree)')
plt.legend()
plt.show()

## 小结

- 变分算法 = 参数化线路 + 经典优化器；问题的全部信息都在哈密顿量里。
- 5 个顶点的 Max-Cut 用暴力枚举瞬间可解：本节的意义在于掌握完整的变分流程，而不是加速。
- 变分算法尚未被证明能带来实用的量子优势：梯度消失、测量开销、可被经典模拟都是绕不开的问题。评价结果时，始终和经典方法比较。